In [ ]:
# Lab type: prompt
# Course: EDA — Exploratory Data Analysis
# Lesson: Directing a Full AI-Assisted EDA
# Task: Write a prompt to direct an AI tool to perform EDA on the orders dataset.
#       Paste the output into the designated cell, then audit it against the
#       six-item checklist. Your goal: catch every issue before any downstream
#       analysis relies on it.

## Setup

Run this cell first. It installs dependencies and generates a synthetic orders dataset.

In [ ]:
!pip install pandas matplotlib seaborn numpy --quiet

import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 50_000

# ── Channels and regions ─────────────────────────────────────────────────
channels = rng.choice(
    ["online", "retail", "wholesale", "enterprise"],
    size=n, p=[0.48, 0.33, 0.14, 0.05]
)
regions_8 = [
    "North", "South", "East", "West",
    "Northeast", "Northwest", "Southeast", "Southwest",
]
region_raw = rng.choice(regions_8 + [None], size=n,
                        p=[0.14, 0.13, 0.14, 0.12, 0.12, 0.11, 0.11, 0.11, 0.02])

# ── Status (6 values) ────────────────────────────────────────────────────
status_probs = {
    "online":     [0.68, 0.12, 0.08, 0.07, 0.03, 0.02],
    "retail":     [0.72, 0.11, 0.07, 0.06, 0.03, 0.01],
    "wholesale":  [0.74, 0.10, 0.07, 0.05, 0.03, 0.01],
    "enterprise": [0.73, 0.11, 0.08, 0.04, 0.03, 0.01],
}
statuses = ["delivered", "shipped", "pending", "cancelled", "returned", "processing"]
order_status = np.empty(n, dtype=object)
for ch, probs in status_probs.items():
    mask = channels == ch
    order_status[mask] = rng.choice(statuses, size=mask.sum(), p=probs)

# ── Dates (stored as strings — need conversion) ───────────────────────────
base_date = pd.Timestamp("2022-01-01")
order_days = rng.integers(0, 730, size=n)
order_dates = [str((base_date + pd.Timedelta(days=int(d))).date()) for d in order_days]
delivery_lag = rng.integers(2, 14, size=n)
delivery_dates = []
for i, (od, st) in enumerate(zip(order_dates, order_status)):
    if st in ("pending", "processing"):
        delivery_dates.append(None)
    else:
        d = pd.Timestamp(od) + pd.Timedelta(days=int(delivery_lag[i]))
        delivery_dates.append(str(d.date()))

# ── Products (concentrated cardinality) ──────────────────────────────────
n_products = 500
product_weights = np.zeros(n_products)
product_weights[:50] = 0.60   # top 10% of SKUs = 60% of orders
product_weights[50:] = 0.40 / 450
product_weights[:50] /= 50
product_weights = product_weights / product_weights.sum()
product_ids_pool = [f'PROD-{i:04d}' for i in range(1, n_products + 1)]
product_ids = rng.choice(product_ids_pool, size=n, p=product_weights)

# ── Quantity ─────────────────────────────────────────────────────────────
quantity = rng.integers(1, 8, size=n).astype(float)
quantity[channels == "wholesale"] = rng.integers(10, 150, size=(channels == "wholesale").sum())
quantity[channels == "enterprise"] = rng.integers(25, 500, size=(channels == "enterprise").sum())
quantity = quantity.astype(int)

# ── Unit price (varies by channel) ───────────────────────────────────────
channel_price_range = {"online": (12, 85), "retail": (15, 120), "wholesale": (8, 45), "enterprise": (50, 300)}
unit_price = np.empty(n)
for ch, (lo, hi) in channel_price_range.items():
    mask = channels == ch
    unit_price[mask] = rng.uniform(lo, hi, size=mask.sum())
unit_price = unit_price.round(2)

# ── Discount ─────────────────────────────────────────────────────────────
discount = rng.beta(1.5, 8, size=n).round(3)
discount[channels == "enterprise"] = rng.beta(3, 5, size=(channels == "enterprise").sum()).round(3)

# ── Shipping cost (correlated with quantity, target ~0.38) ───────────────
shipping_cost = (rng.uniform(3, 12, size=n) + quantity * rng.uniform(0.05, 0.35, size=n)).round(2)
shipping_cost = np.clip(shipping_cost, 2.0, 250.0)

# ── Return flag (deliberately messy: mix of 'True'/'False' and 'Yes'/'No') 
return_raw = []
for st in order_status:
    if st == "returned":
        return_raw.append(rng.choice(["True", "Yes"]))
    else:
        return_raw.append(rng.choice(["False", "No"]))
return_flag = np.array(return_raw)

# ── Revenue ──────────────────────────────────────────────────────────────
revenue = (unit_price * quantity * (1 - discount) - shipping_cost).round(2)
revenue = np.maximum(revenue, 0.5)

# ── Assemble DataFrame ───────────────────────────────────────────────────
order_ids = [f'ORD-{i:06d}' for i in range(1, n + 1)]
customer_ids = [f'CUST-{rng.integers(1, 10000):05d}' for _ in range(n)]

df = pd.DataFrame({
    "order_id":      order_ids,
    "customer_id":   customer_ids,
    "order_date":    order_dates,
    "product_id":    product_ids,
    "quantity":      quantity,
    "unit_price":    unit_price,
    "discount":      discount,
    "shipping_cost": shipping_cost,
    "region":        region_raw,
    "channel":       channels,
    "status":        order_status,
    "delivery_date": delivery_dates,
    "return_flag":   return_flag,
    "revenue":       revenue,
})

print(f'Dataset: {len(df):,} rows × {len(df.columns)} columns')
print()
print('Column dtypes (note object columns that need conversion):')
print(df.dtypes.to_string())
print()
df.head(3)

---
## The Task

You are preparing an EDA for a churn prediction project. The analysis goal is to **understand revenue drivers by channel and region** — which segments generate the most revenue, how spread out that revenue is, and what relationships exist between numeric columns.

You will:
1. Write a precise prompt for an AI tool (Claude, GPT-4, Copilot — your choice)
2. Review the generated code before running it
3. Paste and run the output
4. Audit it against the checklist below

The dataset has deliberate characteristics that naive prompts typically miss. The audit checklist tells you exactly what to look for.

---
## Phase 1 — Brief the AI

Write a prompt that specifies the column names and types, the analysis goal, known data quality issues, and what to include. Compare your prompt to the *weak* and *better* examples in the lesson — specificity is the variable.

In [ ]:
# PROMPT ENTRY
#
# Write your prompt in the string below, then copy it into your AI tool.
# Include at minimum:
#   - All 14 column names and their current dtypes (copy from the dtypes output above)
#   - Which columns need type conversion (order_date, delivery_date → datetime;
#     return_flag → bool)
#   - The analysis goal: revenue drivers by channel and region
#   - What to include: (1) type conversion first, (2) missing value profile,
#     (3) distributions for quantity, unit_price, revenue, (4) median AND mean
#     revenue by channel with count and std, (5) correlation matrix
#   - Ask for a comment before each section explaining what it checks and why

PROMPT = """
[Write your prompt here]
"""

print(PROMPT)

<details>
<summary>🔑 Model prompt — Phase 1</summary>

**Example strong prompt:**

> I have a pandas DataFrame `df` with 50,000 rows and 14 columns. Column names and current dtypes:
> order_id (object), customer_id (object), order_date (object — convert to datetime),
> product_id (object), quantity (int64), unit_price (float64), discount (float64),
> shipping_cost (float64), region (object — ~2% nulls), channel (object),
> status (object), delivery_date (object — convert to datetime; null for pending/processing orders),
> return_flag (object — mixed strings 'True'/'Yes'/'False'/'No' — convert to bool),
> revenue (float64).
>
> Write Python EDA code that:
> 1. **First block only:** Convert order_date and delivery_date to datetime, return_flag to bool — before any other analysis.
> 2. Profile missing values on the original DataFrame (do NOT drop any rows before this step).
> 3. Plot histograms for quantity, unit_price, and revenue.
> 4. Compute median AND mean revenue by channel, with count and std per group.
> 5. Produce a Pearson correlation matrix for the five numeric columns.
> Add a comment before each section explaining what it checks and why. Use correlational language in any narrative (e.g. "is associated with", not "drives" or "causes").

**Why it's strong:** It specifies all 14 column names with dtypes, calls out which columns need conversion and when (first block, before any analysis), lists the five required output sections in order, and explicitly asks for median not just mean. This leaves no room for the AI to apply its own defaults silently.

</details>

---
## Phase 2 — Review the Code Before Running

Read the AI output before pasting it below. Fix any issues you find now — running broken code produces confusing intermediate results.

In [ ]:
# PRE-RUN REVIEW CHECKLIST
#
# Check the AI output for these four issues before running it.
# Fix anything marked [✗] before you paste into Phase 3.
#
#  [ ]  Type conversions (order_date, delivery_date, return_flag) are in the
#        FIRST code block — before any .describe(), .corr(), or .groupby()
#
#  [ ]  No .dropna() or row-dropping step appears before the missing value
#        count section
#
#  [ ]  Revenue groupby summaries include median (not mean alone)
#
#  [ ]  No mid-script subset filter (df = df[df[...] == ...]) without an
#        inline comment stating the scope it creates
#
# Pre-run notes (what you found and changed):
# ...

---
## Phase 3 — Paste and Run the AI Output

Paste the AI-generated EDA code here (replacing the comment block below) and run it. The dataset is already loaded as `df`.

In [ ]:
# ── PASTE AI OUTPUT BELOW ────────────────────────────────────────────────
#
# Replace this comment block with the AI-generated code.
# The DataFrame is already available as `df`.
#
# ─────────────────────────────────────────────────────────────────────────

---
## Phase 4 — Structured Audit

Review the output you just ran against each criterion. For every item, mark `[ ]` → `[✓]` (pass) or `[✗]` (fail) and add a one-line note.

In [ ]:
# AUDIT CHECKLIST
# ─────────────────────────────────────────────────────────────────────────
#
# 1. TYPE CONVERSION ORDER
#    [ ] order_date, delivery_date, and return_flag are converted before
#        any call to .describe(), .corr(), or .groupby()
#    Note:
#
# 2. MISSING VALUE PROFILING
#    [ ] Missing value counts are reported on the original DataFrame —
#        no .dropna() or row-dropping step appears before this section
#    Note:
#
# 3. REVENUE AGGREGATION
#    [ ] Revenue groupby summaries report median (or both mean and median)
#        — mean is not used as the sole measure for a right-skewed column
#    Note:
#
# 4. SUBSET FILTER DOCUMENTATION
#    [ ] Any mid-script subset filter (df = df[df[...] == ...]) is
#        accompanied by a comment stating the scope it creates
#    Note:
#
# 5. GROUP SUMMARY COMPLETENESS
#    [ ] Every groupby summary that reports a central tendency measure
#        also includes a count column
#    Note:
#
# 6. CAUSAL LANGUAGE
#    [ ] Narrative or summary text uses correlational language
#        ('is associated with', 'correlates with') — causal verbs
#        ('drives', 'causes', 'explains') do not appear
#    Note:
#
# ─────────────────────────────────────────────────────────────────────────
# FINDINGS LOG
#
# For each [✗] item, record:
#   Category: Incorrect finding | Incomplete finding | Missing analysis
#   Issue:    what the AI got wrong or omitted
#   Fix:      write the corrected code in the cell below
#
# ITEM 1:
# ITEM 2:
# ITEM 3:
# ...

---
## Phase 5 — Record Fixes

For each `[✗]` item in your audit, write the corrected or missing code below. Label each block with the audit item number it addresses.

In [ ]:
# FIXES AND ADDITIONS
# Label each block with the audit item number (e.g. '# --- Item 3 fix ---')

# --- Item X fix ---

In [ ]:
# INSTRUCTOR NOTE
# Expected failure modes from naive AI prompts for this task.
# Use this to verify the lab is surfacing the intended issues.
#
# 1. TYPE CONVERSION ORDER (very common)
#    A prompt that omits column types typically generates df.describe() in
#    the first block before any conversions. order_date and delivery_date
#    (stored as object/string) are silently excluded from describe() and
#    corr(). return_flag (stored as mixed 'True'/'False'/'Yes'/'No' strings)
#    is never converted to bool — it appears as an object column with 2
#    cardinalities rather than a boolean.
#
# 2. MEAN FOR REVENUE (very common)
#    Revenue is right-skewed: enterprise orders pull the mean ~40% above the
#    median. AI tools default to .mean() for every aggregation. A student who
#    accepts the AI output without checking this will overstate the typical
#    revenue for the enterprise channel.
#
# 3. DROPNA BEFORE PROFILING (common)
#    ~5% of rows have null region; ~3% have null delivery_date (status =
#    'pending' or 'processing'). AI tools often apply .dropna() at the top.
#    The structural pattern — null delivery_date correlates with pending/
#    processing status — is analytically meaningful and is lost if rows are
#    dropped before missingness is profiled.
#
# 4. CAUSAL LANGUAGE (consistent)
#    AI narrative summaries almost always include causal verbs: 'discount
#    drives revenue loss', 'channel explains the revenue gap', 'quantity
#    causes higher shipping cost'. These are correlational at best.
#
# 5. MISSING ANALYSIS (varies by prompt quality)
#    Well-specified prompts may still omit:
#    - return_flag analysis by channel (return rates differ meaningfully)
#    - Delivery lag (delivery_date - order_date) as a derived feature
#    - Scatter plot of quantity vs shipping_cost coloured by channel
#      (the ~0.38 correlation hides a wholesale-driven pattern)
#    - Cardinality check on product_id (top 10% of SKUs account for 60%
#      of orders — a fact worth surfacing before feature engineering)

<details>
<summary>🔑 Instructor notes — expected failure modes</summary>

**1. Type conversion order (very common):** A prompt that omits column types typically generates `df.describe()` in the first block before any conversions. `order_date` and `delivery_date` are silently excluded from `describe()` and `corr()`. `return_flag` (stored as mixed `'True'/'False'/'Yes'/'No'` strings) appears as an object column with 4 cardinalities rather than a boolean.

**2. Mean for revenue (very common):** Revenue is right-skewed — enterprise orders pull the mean ~40% above the median. AI tools default to `.mean()` for every aggregation. Accepting the AI output without checking this overstates the typical enterprise revenue.

**3. dropna before profiling (common):** ~5% of rows have null `region`; ~3% have null `delivery_date` (status = pending/processing). AI tools often apply `.dropna()` at the top. The structural pattern — null `delivery_date` correlates with pending/processing status — is lost if rows are dropped before missingness is profiled.

**4. Causal language (consistent):** AI narrative summaries almost always use causal verbs: "discount drives revenue loss", "channel explains the revenue gap". These are correlational at best.

**5. Missing analysis (varies by prompt quality):** Well-specified prompts may still omit: return_flag analysis by channel; delivery lag as a derived feature; scatter of quantity vs shipping_cost coloured by channel (the r = 0.38 hides a wholesale-driven pattern); cardinality check on product_id (top 10% of SKUs = 60% of volume).

</details>

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Strong prompt elements:** Include all column names with dtypes, specify type conversion must be the first block, list exactly which sections to produce, ask for median alongside mean, request correlational (not causal) language.

2. **Most common AI failure modes:** Placing `describe()` before type conversion, using `.mean()` as the sole revenue aggregation, and applying `.dropna()` before the missing value profile — all preventable with a well-specified prompt.

</details>